# Games and explicit partial stages

A small executable sketch of the [game algebra](../docs/design/game-algebra.md). `Partial` supplies only an abstract call/map interface. The concrete partial families and three-player table live in this notebook. Follow the trace to see exactly when each stage runs.

In [ ]:
from collections.abc import Callable
from dataclasses import dataclass
from typing import Any, ClassVar

import jax
import jax.numpy as jnp
from jaxtyping import Float, Shaped, TypeCheckError, UInt8
from ndlike.protected_axis.jax import JaxAxisProtected

from shapiq.game import Game, PackedFinitePlayerGame
from shapiq.partial import Partial
from shapiq.players import FinitePlayers
from shapiq.typing import typechecked

## Runtime shape contracts on concrete objects

Jaxtyping field annotations also become checked dataclass-constructor arguments. `*batch` anchors a logical batch; `#*batch` permits broadcasting into it. The table length is `2**n_players`, derived from the constructor's `_players` argument. Both constructors and the public methods use shapiq's `@typechecked`: jaxtyping with beartype at runtime and in normal tests. Opting in with `pytest --typeguard` switches these boundaries to exhaustive typeguard checking and installs an import hook for functions without an explicit decorator.

Evaluation lets the game and coalition batches broadcast together, then requires the returned values to have exactly that combined batch shape. `is_valid` remains a predicate: unsupported coalition shapes or dtypes return `False`. We use normal Python annotations rather than the stringifying `from __future__ import annotations`; Python 3.14+ evaluates these lazily without that import. Restart the kernel before running all cells if it previously used the future import.

`FinitePlayers` now lives in `shapiq.players`. Construct it from an ID array, a metadata catalog/list, both, or just an integer player count, positionally or with `ids=`, `players=`, and `n_players=`. Metadata or a count alone generates sequential IDs and batch shape `()`. Its `ids` array has shape `(*batch, n_players)` and is sorted at construction, so coalition positions follow ascending ID order. Raw metadata is wrapped in a shared, read-only `LatentPlayers` sequence; existing catalogs are reused. Array-like backings retain their NumPy, JAX, or ndlike representation. Lists always become 1D NumPy object arrays, preserving each entry by reference without automatic packing or dtype conversion. Integer access returns a player; slices and boolean filters return catalogs. Sets in a batch can reference different catalog entries, provided they have the same number of players. `is_valid` accepts both boolean membership arrays and packed `uint8` coalitions.

`FinitePlayerGame` validates coalitions and dispatches evaluation by dtype. Choose `BoolFinitePlayerGame` to implement `_eval_bool`, or `PackedFinitePlayerGame` to implement `_eval_packed`; the chosen base supplies the conversion for the other representation. For native support of both formats, implement both hooks directly on `FinitePlayerGame`. The table uses `PackedFinitePlayerGame` and implements only `_eval_packed`, combining little-endian bytes directly into an integer table index without unpacking. It uses `uint32` by default and `uint64` when JAX x64 is enabled; enabling x64 is not necessary for this toy table.

In [11]:
@typechecked
@dataclass
class TableGame(
    PackedFinitePlayerGame[Any, Float[jax.Array, "..."]], JaxAxisProtected[jax.Array]
):
    values: Float[jax.Array, "*batch 2**{_players.ids.shape[-1]}"]
    _players: Shaped[FinitePlayers, "#*batch"]
    protected_axes: ClassVar[dict[str, int]] = {"values": 1}

    @classmethod
    def _pytree_child_names(cls) -> tuple[str, ...]:
        # Players are dynamic, but not an equally batched protected field.
        return ("values", "_players")

    @property
    @typechecked
    def players(self: Shaped[Game, "*batch"]) -> Shaped[FinitePlayers, "#*batch"]:
        return self._players

    @typechecked
    def _eval_packed(
        self: Shaped[Game, "#*batch"],
        coalition: UInt8[jax.Array, "#*batch ({self.players.n_players}+7)//8"],
    ) -> Float[jax.Array, "*batch"]:
        n_players = self.players.n_players
        index_dtype = jnp.uint64 if jax.config.x64_enabled else jnp.uint32
        if n_players > jnp.iinfo(index_dtype).bits:
            msg = "Table indices exceed the enabled integer width."
            raise ValueError(msg)
        shifts = 8 * jnp.arange(coalition.shape[-1], dtype=index_dtype)
        index = jnp.sum(coalition.astype(index_dtype) << shifts, axis=-1, dtype=index_dtype)
        index &= jnp.array((1 << n_players) - 1, dtype=index_dtype)
        shape = jnp.broadcast_shapes(self.shape, index.shape)
        table = jnp.broadcast_to(self.values, (*shape, self.values.shape[-1]))
        index = jnp.broadcast_to(index, shape)
        return jnp.take_along_axis(table, index[..., None], axis=-1)[..., 0]

In [12]:
coalitions = jnp.array(
    [
        [0, 0, 0],  # empty
        [1, 0, 0],  # {0}
        [0, 1, 0],  # {1}
        [1, 1, 0],  # {0, 1}
        [0, 0, 1],  # {2}
        [1, 0, 1],  # {0, 2}
        [0, 1, 1],  # {1, 2}
        [1, 1, 1],  # {0, 1, 2}
    ],
    dtype=bool,
)
g = TableGame(jnp.array([[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0]]), FinitePlayers(3))
print("game batch:", g.shape, "shared players:", g.players.shape)
for coalition, value in zip(coalitions.tolist(), g(coalitions).tolist(), strict=True):
    print(coalition, "->", value)

game batch: (1,) shared players: ()
[False, False, False] -> 0.0
[True, False, False] -> 1.0
[False, True, False] -> 2.0
[True, True, False] -> 4.0
[False, False, True] -> 3.0
[True, False, True] -> 5.0
[False, True, True] -> 6.0
[True, True, True] -> 10.0


## Watch invalid contracts fail

These failures come from the annotated constructor/evaluation boundaries, not manually duplicated assertions. The test suite executes this notebook and also checks valid broadcasting, zero-player tables, and evaluation return shapes.

In [13]:
invalid_examples = [
    ("floating player IDs", lambda: FinitePlayers(jnp.arange(3, dtype=jnp.float32))),
    ("missing player axis", lambda: FinitePlayers(jnp.array(3))),
    ("integer table values", lambda: TableGame(jnp.arange(8)[None, :], g.players)),
    ("wrong table length", lambda: TableGame(jnp.zeros((1, 7)), g.players)),
    (
        "players enlarge game batch",
        lambda: TableGame(jnp.zeros((1, 8)), FinitePlayers(jnp.zeros((2, 3), dtype=int))),
    ),
    ("wrong coalition dtype", lambda: g.eval(jnp.zeros((3,)))),
    ("wrong coalition width", lambda: g.eval(jnp.zeros((4,), dtype=bool))),
]
for description, attempt in invalid_examples:
    try:
        attempt()
    except TypeCheckError:
        print(f"{description}: rejected")
    else:
        msg = f"Shape contract was not enforced: {description}"
        raise AssertionError(msg)

floating player IDs: rejected
missing player axis: rejected
integer table values: rejected
wrong table length: rejected
players enlarge game batch: rejected
wrong coalition dtype: rejected
wrong coalition width: rejected


## Each partial family owns its call signature and map

`Factors[T]` accepts scale factors; `Offset[T]` accepts an offset. Each implements its own `map`, preserving its family and deferring the callback. `make_family(g)` returns `Factors[Offset[TableGame]]`: factors must be supplied before the offset. Transformations are ordinary functions; nested mapping selects their target explicitly. There is no partial shape/dtype tracking or automatic traversal.

In [14]:
trace = []


@dataclass(frozen=True)
class Factors[T](Partial[T]):
    function: Callable[[jax.Array], T]

    def __call__(self, *, factors: jax.Array) -> T:
        return self.function(factors)

    def map[U](self, function: Callable[[T], U]) -> "Factors[U]":
        return Factors(lambda factors: function(self(factors=factors)))


@dataclass(frozen=True)
class Offset[T](Partial[T]):
    function: Callable[[float], T]

    def __call__(self, *, offset: float) -> T:
        return self.function(offset)

    def map[U](self, function: Callable[[T], U]) -> "Offset[U]":
        return Offset(lambda offset: function(self(offset=offset)))


def double_game(game: TableGame) -> TableGame:
    trace.append("double")
    return TableGame(2 * game.values, game.players)


def make_family(game: TableGame) -> Factors[Offset[TableGame]]:
    assert game.shape == (1,)  # this toy creator starts with a one-game batch

    def bind_factors(factors: jax.Array) -> Offset[TableGame]:
        trace.append("bind factors")

        def bind_offset(offset: float) -> TableGame:
            trace.append("materialize")
            values = jnp.asarray(factors)[:, None] * game.values + offset
            return TableGame(values, game.players)

        return Offset(bind_offset)

    return Factors(bind_factors)

In [15]:
trace.clear()
direct = double_game(g)
assert trace == ["double"]
assert bool(jnp.allclose(direct.eval(coalitions), 2 * g(coalitions)))
trace.clear()
graph = make_family(g).map(lambda offset: offset.map(double_game))
print("unbound:", type(graph).__name__, trace)
assert isinstance(graph, Factors) and trace == []

remaining = graph(factors=jnp.array([1.0, 2.0]))
print("one stage bound:", type(remaining).__name__, trace)
assert isinstance(remaining, Offset)
assert trace == ["bind factors"]

result = remaining(offset=0.5)
print("fully bound:", result.shape, trace)
assert result.shape == (2,) and trace == ["bind factors", "materialize", "double"]

# Add a singleton batch axis to evaluate every coalition for every game.
values = result[:, None](coalitions)
print(values)
assert bool(jnp.allclose(values, 2 * (jnp.array([1.0, 2.0])[:, None] * g.values + 0.5)))

unbound: Factors []
one stage bound: Offset ['bind factors']
fully bound: (2,) ['bind factors', 'materialize', 'double']
[[ 1.  3.  5.  9.  7. 11. 13. 21.]
 [ 1.  5.  9. 17. 13. 21. 25. 41.]]


## Explicit evaluation and callbacks that introduce stages

Evaluation is an ordinary callback mapped to the game at the chosen depth. A callback may also return another partial: that new stage is kept intact, even if its parameter has the same name as an earlier stage. Mapping does not execute either stage.

In [16]:
trace.clear()
pending_values = make_family(g).map(
    lambda offset: offset.map(lambda game: double_game(game).eval(coalitions[0]))
)
assert trace == []
values = pending_values(factors=jnp.array([1.0, 2.0]))(offset=0.5)
assert trace == ["bind factors", "materialize", "double"]
assert bool(jnp.allclose(values, 1.0))
print("deferred values:", values, trace)


def add_offset(game: TableGame) -> Offset[TableGame]:
    return Offset(lambda offset: TableGame(game.values + offset, game.players))


trace.clear()
extended = make_family(g).map(lambda offset: offset.map(add_offset))
assert trace == []
extra = extended(factors=jnp.array([1.0, 2.0]))(offset=0.5)
assert isinstance(extra, Offset)
assert trace == ["bind factors", "materialize"]
result = extra(offset=1.0)
assert bool(jnp.allclose(result.values, jnp.array([1.0, 2.0])[:, None] * g.values + 1.5))

deferred values: [1. 1.] ['bind factors', 'materialize', 'double']


## A handler whose input is itself partial

The target need not be a concrete game. Mapping only over the outer `Factors` stage lets a handler receive the inner `Offset` intact. Here it repackages that stage so the eventual offset is doubled, without materializing a game yet.

In [17]:
def replace_offset(stage: Offset[TableGame]) -> Offset[TableGame]:
    trace.append("replace offset")
    return Offset(lambda offset: stage(offset=2 * offset))


trace.clear()
changed = make_family(g).map(replace_offset)
assert trace == []
after_factors = changed(factors=jnp.array([1.0, 2.0]))
assert isinstance(after_factors, Offset)
assert trace == ["bind factors", "replace offset"]
print("new stage awaits offset:", type(after_factors).__name__, trace)
result = after_factors(offset=0.5)
assert trace == ["bind factors", "replace offset", "materialize"]
assert bool(jnp.allclose(result.values, jnp.array([1.0, 2.0])[:, None] * g.values + 1.0))
print("result:", result.values)

new stage awaits offset: Offset ['bind factors', 'replace offset']
result: [[ 1.  2.  3.  5.  4.  6.  7. 11.]
 [ 1.  3.  5.  9.  7. 11. 13. 21.]]
